In [1]:

# import necessary modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

!pip -qq install pyreadstat
!pip -qq install xgboost
import requests
import pyreadstat
from pathlib import Path

import zipfile
import re
import os
import pickle
from google.colab import drive
import matplotlib.ticker as mtick

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 18.9 MB/s eta 0:00:00


In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# create folder to hold data
data_path = Path(f'/content/drive/MyDrive/CIS-545-Final-Project/data/data_{year}')
fig_path = data_path / "figures"

data_path.mkdir(parents=True, exist_ok=True)
fig_path.mkdir(parents=True, exist_ok=True)

metrics = {}

In [5]:
!wget -nc https://www.cdc.gov/brfss/annual_data/{year}/files/LLCP{year}XPT.zip

--2026-04-15 21:06:14--  https://www.cdc.gov/brfss/annual_data/2023/files/LLCP2023XPT.zip
Resolving www.cdc.gov (www.cdc.gov)... 23.194.148.226, 2600:1407:7400:193::2461, 2600:1407:7400:196::2461
Connecting to www.cdc.gov (www.cdc.gov)|23.194.148.226|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 93237818 (89M) [application/x-zip-compressed]
Saving to: ‘LLCP2023XPT.zip’

LLCP2023XPT.zip     100%[===================>]  88.92M   108MB/s    in 0.8s    

2026-04-15 21:06:15 (108 MB/s) - ‘LLCP2023XPT.zip’ saved [93237818/93237818]



In [6]:
def create_brfss_df(year):

  data_path = Path(f'/content/LLCP{year}XPT.zip')

  with zipfile.ZipFile(data_path, 'r') as z:
    xpt_files = [f for f in z.namelist() if f.strip().lower().endswith('.xpt')]
    xpt_file = xpt_files[0]

    with z.open(xpt_file) as f:
      brfss_df, metadata = pyreadstat.read_xport(f, encoding='latin1')


  column_map = dict(zip(metadata.column_names, metadata.column_labels))

  return brfss_df, column_map

In [7]:
brfss_df, column_map = create_brfss_df(year)

print(f"There are {brfss_df.shape[1]} columns and {brfss_df.shape[0]} rows.")

brfss_df.to_parquet(data_path / "raw.parquet")

There are 350 columns and 433323 rows.


In [8]:
def create_diabetes_df(brfss_df):

  # dict of target features & their respective column naming convention
  target_feature_map = {
      "index": "ID",
      r"DIABETE\d?" : "Diabetes_Binary",
      r"_RFHYPE\d?" : "HighBP",
      r"_RFCHOL\d?" : "HighChol",
      r"_CHOLCH." : "CholCheckWithinPast5Years",
      "_BMI5" : "BMI",
      "SMOKE100" : "Smoker",
      "CVDSTRK3" : "Stroke",
      "_MICHD" : "HeartDiseaseorAttack",
      "_TOTINDA" : "PhysActivity",
      r"_RFDRHV\d?" : "HvyAlcoholConsump",
      r"_?HLTHPLN?\d?" : "AnyHealthcare",
      r"MEDCOST\d?"  : "NoDocbcCost",
      "GENHLTH" : "GenHlth",
      "MENTHLTH" : "MentHlth",
      "PHYSHLTH" : "PhysHlth",
      "DIFFWALK" : "DiffWalk",
      r"_?SEX$" : "Sex",
      "_AGEG5YR" : "Age",
      "EDUCA" : "Education",
      r"_INCOMG\d?" : "Income",
  }

  extracted_feature_map = {}

  # exact match
  for feature in target_feature_map:
    if column_map.get(feature):
      extracted_feature_map[feature] = target_feature_map[feature]
   #regex match
    else:
      for col in column_map:
        if re.match(feature, col):
          extracted_feature_map[col] = target_feature_map[feature]


  # Create diabetes df
  diabetes_df = brfss_df[list(extracted_feature_map.keys())] \
              .reset_index(drop=True) \
              .rename(columns=extracted_feature_map)

  return diabetes_df

In [9]:
diabetes_df = create_diabetes_df(brfss_df)

In [10]:
# Cleans diabetes df by collapsing surveyed fields into binary & integer encodings
def clean_diabetes_df(diabetes_df):

  def isBinaryFeature(feature):
    return feature in ["HighBP", "HighChol", "Smoker",
                      "Stroke", "HeartDiseaseorAttack", "PhysActivity",
                      "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost",
                      "DiffWalk", "Sex"]

  def isFlipped(feature):
    return feature in ["DiffWalk", "HeartDiseaseorAttack", "Stroke",
                       "Smoker", "PhysActivity",
                       "NoDocbcCost", "AnyHealthcare"]

  def filter_feature (df, feature, max_val):
    return df[df[feature] <= max_val].copy()

  def clean_binary_features(df, feature):
    df = filter_feature(df, feature, 2)
    df[feature] = (df[feature] == 2).astype(int)
    if isFlipped(feature):
      df[feature] = 1 - df[feature]
    return df

  def clean_Diabetes_Binary(df, feature):
    df = filter_feature(df, feature, 4)
    df[feature]= (df[feature].isin([1,2,4])).astype(int)
    return df

  def clean_CholCheckWithinPast5Years(df, feature):
    df = filter_feature(df, feature, 3)
    df[feature] = (df[feature] == 1).astype(int)
    return df

  def clean_BMI(df, feature):
    df = df[df[feature] != 9999].copy()
    df[feature] = df[feature] / 100
    return df

  def clean_GenHlth(df, feature):
    df = filter_feature(df, feature, 5)
    df[feature] = df[feature].astype(int)
    return df

  def clean_MentHlth_and_PhysHlth(df, feature):
    df[feature] = (df[feature].replace(88,0)).astype(int)
    return filter_feature(df, feature, 30)

  def clean_Age(df, feature):
    df = filter_feature(df, feature, 13)
    df[feature] = df[feature].astype(int)
    return df

  def clean_Education(df, feature):
    df = filter_feature(df, feature, 7)
    df[feature] = df[feature].astype(int)
    return df

  def clean_Income(df, feature):
    df = filter_feature(df, feature, 11)
    df[feature] = df[feature].astype(int)
    return df

  non_binary_feature_handlers = {
      "Diabetes_Binary": clean_Diabetes_Binary,
      "CholCheckWithinPast5Years": clean_CholCheckWithinPast5Years,
      "BMI": clean_BMI,
      "GenHlth": clean_GenHlth,
      "MentHlth": clean_MentHlth_and_PhysHlth,
      "PhysHlth": clean_MentHlth_and_PhysHlth,
      "Age": clean_Age,
      "Education": clean_Education,
      "Income": clean_Income,
  }

  diabetes_cleaned_df = diabetes_df.copy().dropna(how='any')

  # Filter out non-valid repsonses for each feature
  for feature in diabetes_cleaned_df:
    if isBinaryFeature(feature):
      # Binarize & Clean binary feature
      diabetes_cleaned_df = clean_binary_features(diabetes_cleaned_df, feature)
    else:
      # Handle filtering and cleaning per non-binary features
      diabetes_cleaned_df = non_binary_feature_handlers.get(feature)(diabetes_cleaned_df, feature)

  diabetes_cleaned_df = diabetes_cleaned_df \
                      .reset_index(drop=True)

  return diabetes_cleaned_df

In [11]:
diabetes_cleaned_df = clean_diabetes_df(diabetes_df)

diabetes_cleaned_df.to_parquet(data_path / "cleaned.parquet")

# **Part 2: Exploratory Data Analysis & Feature Selection**

In [12]:
#Check for the shape
diabetes_cleaned_df.shape

(5000, 20)

In [13]:
#Check for any null value
diabetes_cleaned_df.isnull().sum()

,0
Diabetes_Binary,0
HighBP,0
HighChol,0
CholCheckWithinPast5Years,0
BMI,0
Smoker,0
Stroke,0
HeartDiseaseorAttack,0
PhysActivity,0
HvyAlcoholConsump,0


In [14]:
#Check for target blance
diabetes_cleaned_df["Diabetes_Binary"].value_counts()

,count
Diabetes_Binary,
0,3988
1,1012


In [15]:
#check for the data type and the amount of rows
diabetes_cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Diabetes_Binary            5000 non-null   int64  
 1   HighBP                     5000 non-null   int64  
 2   HighChol                   5000 non-null   int64  
 3   CholCheckWithinPast5Years  5000 non-null   int64  
 4   BMI                        5000 non-null   float64
 5   Smoker                     5000 non-null   int64  
 6   Stroke                     5000 non-null   int64  
 7   HeartDiseaseorAttack       5000 non-null   int64  
 8   PhysActivity               5000 non-null   int64  
 9   HvyAlcoholConsump          5000 non-null   int64  
 10  AnyHealthcare              5000 non-null   int64  
 11  NoDocbcCost                5000 non-null   int64  
 12  GenHlth                    5000 non-null   int64  
 13  MentHlth                   5000 non-null   int64

In [16]:
#Check the describtion of the data
diabetes_cleaned_df.describe()

,Diabetes_Binary,HighBP,HighChol,CholCheckWithinPast5Years,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.0000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,0.202400,0.525800,0.475000,0.967000,29.181492,0.405600,0.060000,0.108600,0.7302,0.061200,0.964800,0.083200,2.723400,3.950200,4.803000,0.202200,0.561800,8.392800,5.034600,5.260000
std,0.401829,0.499384,0.499425,0.178654,6.496403,0.491057,0.237511,0.311168,0.4439,0.239721,0.184303,0.276212,1.043988,8.078142,9.113748,0.401681,0.496216,3.242501,0.968706,2.159568
min,0.000000,0.000000,0.000000,0.000000,13.160000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,1.000000,24.655000,0.000000,0.000000,0.000000,0.0000,0.000000,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,6.000000,4.000000,4.000000
50%,0.000000,1.000000,0.000000,1.000000,28.250000,0.000000,0.000000,0.000000,1.0000,0.000000,1.000000,0.000000,3.000000,0.000000,0.000000,0.000000,1.000000,9.000000,5.000000,5.000000
75%,0.000000,1.000000,1.000000,1.000000,32.610000,1.000000,0.000000,0.000000,1.0000,0.000000,1.000000,0.000000,3.000000,3.000000,5.000000,0.000000,1.000000,11.000000,6.000000,6.000000
max,1.000000,1.000000,1.000000,1.000000,76.820000,1.000000,1.000000,1.000000,1.0000,1.000000,1.000000,1.000000,5.000000,30.000000,30.000000,1.000000,1.000000,13.000000,6.000000,9.000000


In [17]:
fig, ax = plt.subplots(figsize=(6,4))

diabetes_cleaned_df["BMI"].hist(bins=30, ax=ax)
ax.set_title("BMI Distribution")
ax.set_xlabel("BMI")
ax.set_ylabel("Frequency")
fig.tight_layout()
fig.savefig(fig_path / "bmi_distribution.png", dpi=200, bbox_inches="tight")

plt.close(fig)

In [18]:
# We saw that the max BMI is 99.8 which seems like faulty survey, so we will remove rows where BMI is above the 99th percentile
diabetes_cleaned_df["BMI"].quantile(0.99)


np.float64(49.132400000000054)

In [19]:
diabetes_cleaned_df = diabetes_cleaned_df[
    diabetes_cleaned_df["BMI"] <= diabetes_cleaned_df["BMI"].quantile(0.99)
]

In [20]:
diabetes_cleaned_df.shape

(4950, 20)

In [21]:
# Graph our heatmap for correlation
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(18,12))
corr = diabetes_cleaned_df.corr()
sns.heatmap(corr, annot=True)

ax.set_title(f"Correlation Heatmap")
fig.tight_layout()
fig.savefig(fig_path / "correlation_heatmap.png", dpi=200, bbox_inches="tight")

plt.close(fig)

In [22]:
#Check correlation against "Diabetes_Binary"
corr_target = (diabetes_cleaned_df.corr()['Diabetes_Binary'] \
.drop('Diabetes_Binary') \
.sort_values(key=abs, ascending=False)
)


metrics["corr_target"] = corr_target

In [23]:
# diabetic vs non-diabetic BMI distribution
fig, ax = plt.subplots(figsize=(7, 5))

# split the data set by diabetic and non-diabetic
for label, group in diabetes_cleaned_df.groupby('Diabetes_Binary'):
    ax.hist(group['BMI'], bins=40, alpha=0.6,   # divy into enough equal BMI bins to capture the difference
            label='Diabetic' if label == 1 else 'Non-Diabetic',
            color='#C44E52' if label == 1 else '#4C72B0')

ax.set_title("BMI Distribution: Diabetic vs Non-Diabetic")
ax.set_xlabel("BMI")
ax.set_ylabel("Count")
ax.legend()

plt.tight_layout()
fig.savefig(fig_path / "bmi_diabetes_comparison.png", dpi=200, bbox_inches="tight")
plt.close(fig)


# summary statistics
medians = diabetes_cleaned_df.groupby('Diabetes_Binary')['BMI'].median()
print(f"Median BMI (Non-Diabetic): {medians[0]:.1f}")
print(f"Median BMI (Diabetic): {medians[1]:.1f}")
print(f"Difference: {medians[1] - medians[0]:.1f} BMI points")

metrics["bmi_medians"] = {
    "non_diabetic": float(medians[0]),
    "diabetic": float(medians[1])
}

Median BMI (Non-Diabetic): 27.5
Median BMI (Diabetic): 30.6
Difference: 3.2 BMI points


In [24]:
# age vs diabetes rate

# translation key for storing numbers/representing ages
age_labels = {
    1:  '18–24', 2:  '25–29', 3:  '30–34',
    4:  '35–39', 5:  '40–44', 6:  '45–49',
    7:  '50–54', 8:  '55–59', 9:  '60–64',
    10: '65–69', 11: '70–74', 12: '75–79',
    13: '80+'
}

age_diabetes = diabetes_cleaned_df.groupby('Age')['Diabetes_Binary'].mean().reset_index()
age_diabetes['AgeLabel'] = age_diabetes['Age'].map(age_labels)
age_diabetes['DiabetesRate'] = age_diabetes['Diabetes_Binary'] * 100

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    age_diabetes['AgeLabel'],
    age_diabetes['DiabetesRate'],
    marker='o',
    color='steelblue',
    linewidth=2.5,
    markersize=7
)

ax.fill_between(
    range(len(age_diabetes)),
    age_diabetes['DiabetesRate'],
    alpha=0.1,
    color='steelblue'
)


ax.set_xticks(range(len(age_diabetes)))
ax.set_xticklabels(age_diabetes['AgeLabel'], rotation=45, ha='right')
ax.set_xlabel("Age Group")
ax.set_ylabel("Diabetes Prevalence (%)")
ax.set_title("Diabetes Prevalence by Age Group")
ax.grid(axis='y', linestyle='--', alpha=0.5)

fig.tight_layout()
fig.savefig(fig_path / "age_diabetes_rate.png", dpi=200, bbox_inches="tight")
plt.close(fig)

metrics["age_diabetes_summary"] = {
    "lowest_prevalence": float(age_diabetes['DiabetesRate'].min()),
    "lowest_group": str(age_diabetes.loc[age_diabetes['DiabetesRate'].idxmin(), 'AgeLabel']),
    "highest_prevalence": float(age_diabetes['DiabetesRate'].max()),
    "highest_group": str(age_diabetes.loc[age_diabetes['DiabetesRate'].idxmax(), 'AgeLabel']),
    "rate_increase": float(
        age_diabetes['DiabetesRate'].max() - age_diabetes['DiabetesRate'].min()
    )
}

# summary statistics
print(f"Lowest prevalence: {age_diabetes['DiabetesRate'].min():.1f}% " f"at age group {age_diabetes.loc[age_diabetes['DiabetesRate'].idxmin(), 'AgeLabel']}")
print(f"Highest prevalence: {age_diabetes['DiabetesRate'].max():.1f}% " f"at age group {age_diabetes.loc[age_diabetes['DiabetesRate'].idxmax(), 'AgeLabel']}")
print(f"Rate increase from youngest to oldest: " f"{age_diabetes['DiabetesRate'].max() - age_diabetes['DiabetesRate'].min():.1f} percentage points")

Lowest prevalence: 2.7% at age group 25–29
Highest prevalence: 29.3% at age group 75–79
Rate increase from youngest to oldest: 26.6 percentage points


In [25]:
# income vs diabetes rate

# translation key for storing numbers/representing incomes
income_labels = {
    1:  '<$10k',    2:  '$10–15k',  3:  '$15–20k',
    4:  '$20–25k',  5:  '$25–35k',  6:  '$35–50k',
    7:  '$50–75k',  8:  '$75–100k', 9:  '$100–150k',
    10: '$150–200k', 11: '>$200k'
}

income_diabetes = diabetes_cleaned_df.groupby('Income')['Diabetes_Binary'].mean().reset_index()
income_diabetes['DiabetesRate'] = income_diabetes['Diabetes_Binary'] * 100

present_codes = sorted(income_diabetes['Income'].astype(int).unique())
income_labels_dynamic = {k: income_labels[k] for k in present_codes}

# if the top bracket in the data is not 11, relabel it as "X+ i.e 75k+"
max_code = max(present_codes)
if max_code < 11:
    top_label = income_labels[max_code]
    top_label = top_label.replace('–', ' to ')
    income_labels_dynamic[max_code] = f"{top_label}+"

income_diabetes['IncomeLabel'] = income_diabetes['Income'].map(income_labels_dynamic)


fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.bar(
    income_diabetes['IncomeLabel'],
    income_diabetes['DiabetesRate'],
    color='steelblue',
    alpha=0.8,
    edgecolor='white'
)

# color the bar with the highest risk red so it stands out
max_idx = income_diabetes['DiabetesRate'].idxmax()
bar_pos = income_diabetes.index.get_loc(max_idx)
bars[bar_pos].set_color('#C44E52')

ax.set_xticklabels(income_diabetes['IncomeLabel'], rotation=45, ha='right')
ax.set_xlabel("Income Bracket")
ax.set_ylabel("Diabetes Prevalence (%)")
ax.set_title("Diabetes Prevalence by Income Bracket")
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Save to data
fig.tight_layout()
fig.savefig(fig_path / "income_diabetes_rate.png", dpi=200, bbox_inches="tight")
plt.close(fig)

metrics["income_diabetes_summary"] = {
    "highest_risk_bracket": str(income_diabetes.loc[max_idx, 'IncomeLabel']),
    "highest_risk_rate": float(income_diabetes['DiabetesRate'].max()),
    "lowest_risk_bracket": str(income_diabetes.loc[income_diabetes['DiabetesRate'].idxmin(), 'IncomeLabel']),
    "lowest_risk_rate": float(income_diabetes['DiabetesRate'].min()),
    "prevalence_gap": float(
        income_diabetes['DiabetesRate'].max() - income_diabetes['DiabetesRate'].min()
    )
}

metrics["income_diabetes_table"] = income_diabetes

# summary statistics
print(f"Highest risk income bracket: " f"{income_diabetes.loc[max_idx, 'IncomeLabel']} "f"at {income_diabetes['DiabetesRate'].max():.1f}%")
print(f"Lowest risk income bracket: " f"{income_diabetes.loc[income_diabetes['DiabetesRate'].idxmin(), 'IncomeLabel']} "f"at {income_diabetes['DiabetesRate'].min():.1f}%")
print(f"Prevalence gap between lowest and highest income: " f"{income_diabetes['DiabetesRate'].max() - income_diabetes['DiabetesRate'].min():.1f} percentage points")

/tmp/ipykernel_3776/490017585.py:42: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(income_diabetes['IncomeLabel'], rotation=45, ha='right')


Highest risk income bracket: <$10k at 30.3%
Lowest risk income bracket: $50–75k at 9.2%
Prevalence gap between lowest and highest income: 21.1 percentage points


In [26]:
# k means cluster

# remove label
X_unsup = diabetes_cleaned_df.drop("Diabetes_Binary", axis=1)
y_true  = diabetes_cleaned_df["Diabetes_Binary"]

scaler_unsup   = StandardScaler()
X_scaled_unsup = scaler_unsup.fit_transform(X_unsup)

# determine k
distortions = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    km.fit(X_scaled_unsup)
    distortions.append(km.inertia_)

#save elbow plot
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(K_range, distortions, marker='o', color='steelblue')
ax.set_xlabel("Number of Clusters (K)")
# format y axis to show full numbers cleanly
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.set_ylabel("Distortion (Inertia)")
ax.set_title("K-Means Elbow Method: Choosing Optimal K")
ax.set_xticks(K_range)

fig.tight_layout()
fig.savefig(fig_path / "kmeans_elbow.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# k means
K_chosen = 3  # cosen after determining no clear elbow

kmeans         = KMeans(n_clusters=K_chosen, init='k-means++', random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled_unsup)

# see if clusters align with diabetes
cluster_analysis = pd.DataFrame({
    'Cluster'         : cluster_labels,
    'Diabetes_Binary' : y_true.values
})

cluster_diabetes_rate = (
    cluster_analysis.groupby("Cluster")["Diabetes_Binary"]
    .mean()
    .round(3)
)

print("\nDiabetes rate per cluster:")
print(cluster_diabetes_rate)

# cluster display
cluster_profile = X_unsup.copy()
cluster_profile['Cluster'] = cluster_labels
cluster_profile = cluster_profile.groupby('Cluster').mean()


# Save heatmap to data
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(cluster_profile, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
ax.set_title("K-Means Cluster Profiles: Average Feature Values per Cluster")

fig.tight_layout()
fig.savefig(fig_path / "kmeans_cluster_profiles.png", dpi=200, bbox_inches="tight")
plt.close(fig)

metrics["kmeans"] = {
    "K_range": K_range,
    "distortions": distortions,
    "K_chosen": K_chosen,
    "cluster_diabetes_rate": cluster_diabetes_rate,
    "cluster_profile": cluster_profile
}


Diabetes rate per cluster:
Cluster
0    0.362
1    0.142
2    0.136
Name: Diabetes_Binary, dtype: float64


In [27]:
# t-SNE visualization


from sklearn.manifold import TSNE
from matplotlib.patches import Patch


# lower sample data set for faster output

MAX_TSNE_SAMPLES = 10000
n_samples = min(MAX_TSNE_SAMPLES, len(X_scaled_unsup))
print(f"Sampling {n_samples} for t-SNE")

sample_idx    = np.random.default_rng(42).choice(len(X_scaled_unsup), size=n_samples, replace=False)
X_tsne_input  = X_scaled_unsup[sample_idx]
tsne_labels   = y_true.values[sample_idx]
tsne_clusters = cluster_labels[sample_idx]


# compress features and run t-sne

tsne   = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_tsne_input)

# combines the 2D coordinates with each patient's diabetes status and K-Means cluster assignment
tsne_df = pd.DataFrame({
    'x'              : X_tsne[:, 0],
    'y'              : X_tsne[:, 1],
    'Diabetes_Label' : tsne_labels,
    'KMeans_Cluster' : tsne_clusters
})

# plots side by side

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# plot 1: actual diabetes label
scatter1 = axes[0].scatter(
    tsne_df['x'], tsne_df['y'],
    c=tsne_df['Diabetes_Label'],
    cmap='coolwarm', alpha=0.4, s=5
)
axes[0].set_title("t-SNE: Colored by Actual Diabetes Label", fontsize=12)
axes[0].set_xlabel("t-SNE Dimension 1")
axes[0].set_ylabel("t-SNE Dimension 2")
fig.colorbar(scatter1, ax=axes[0], label='0 = No Diabetes     1 = Diabetes')

# define 3 explicit colors, one per cluster
cluster_colors = {0: '#4C72B0', 1: '#29B8A0', 2: '#8B4513'}
colors = [cluster_colors[c] for c in tsne_df['KMeans_Cluster']]

axes[1].scatter(
    tsne_df['x'], tsne_df['y'],
    c=colors,
    alpha=0.4, s=5
)

# manually build a clean legend
legend_elements = [
    Patch(facecolor='#4C72B0', label='Cluster 0'),
    Patch(facecolor='#29B8A0', label='Cluster 1'),
    Patch(facecolor='#8B4513', label='Cluster 2'),
]
axes[1].legend(handles=legend_elements, loc='lower right')

axes[1].set_title(f"t-SNE: Colored by K-Means Cluster (K={K_chosen})", fontsize=12)
axes[1].set_xlabel("t-SNE Dimension 1")
axes[1].set_ylabel("t-SNE Dimension 2")

plt.suptitle(
    "Do K-Means clusters align with actual diabetes outcomes?",
    fontsize=13, y=1.02
)

fig.savefig(fig_path / "tsne_kmeans_comparison.png", dpi=200, bbox_inches="tight")
plt.close(fig)

metrics["tsne"] = {
    "n_samples": int(n_samples),
    "perplexity": 30,
    "n_iter": 1000,
    "k_chosen": int(K_chosen)
}

Sampling 4950 for t-SNE


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


# **Part 3: Application of Statistical and Machine Learning Techniques**


In [28]:
# Perform scaling and split dataset into training and test sets

def scale_and_split(diabetes_df):

  from sklearn.model_selection import train_test_split
  X = diabetes_df.drop("Diabetes_Binary", axis=1)
  y = diabetes_df["Diabetes_Binary"]
  X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=0.2, random_state=42
  )
  scaler = StandardScaler()

  X_train_scaled = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)

  return X_train, X_train_scaled, X_test_scaled, y_train, y_test

X_train, X_train_scaled, X_test_scaled, y_train, y_test = scale_and_split(diabetes_cleaned_df)

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix

# Define functions to handle modeling for Random Forest, K-Nearest Neighbors, Linear Regression, and XGBoost
def get_model(model_choice, y_train):

    models = {
        "Random Forest": lambda: RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ),
        "KNN": lambda: KNeighborsClassifier(
            n_neighbors=5
        ),
        "Logistic Regression": lambda: LogisticRegression(
            class_weight="balanced",
            random_state=42,
            max_iter=1000
        ),
        "XGBoost": lambda: XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=5,
            scale_pos_weight = ((len(y_train) - y_train.sum()) / y_train.sum()),
            random_state=42,
            n_jobs=-1
        )
    }

    return models[model_choice]()

# Evaluates model performance and creates confusion matrix

def train_and_evaluate_model(model_choice, X_train_scaled, X_test_scaled, y_train, y_test):

  model = get_model(model_choice, y_train)

  model.fit(X_train_scaled, y_train)

  y_pred = model.predict(X_test_scaled)

  cm = confusion_matrix(y_test, y_pred)

  return model, y_pred, cm


In [30]:
# Train and evaluate all selected models on all features

# Using Random Forest - Train and Evaluate RF on all Features
rf_model, rf_y_pred, rf_cm  = train_and_evaluate_model("Random Forest", X_train_scaled, X_test_scaled, y_train, y_test)

# Using K-Nearest Neighbors - Train and Evaluate KNN on all Features
knn_model, knn_y_pred, knn_cm = train_and_evaluate_model("KNN", X_train_scaled, X_test_scaled, y_train, y_test)

# Using Logistic Regression - Train and Evaluate KNN on all Features
lr_model, lr_y_pred, lr_cm = train_and_evaluate_model("Logistic Regression", X_train_scaled, X_test_scaled, y_train, y_test)

# Using XGBoost - Train and Evaluate KNN on all Features
xgb_model, xgb_y_pred, xgb_cm = train_and_evaluate_model("XGBoost", X_train_scaled, X_test_scaled, y_train, y_test)


# Plot each model's confusion matrix side by side
fig, axes = plt.subplots(1, 4, figsize=(20,5))

models = [
    ("Random Forest", rf_cm),
    ("KNN", knn_cm),
    ("Logistic Regression", lr_cm),
    ("XGBoost", xgb_cm)
]

for ax, (name, cm) in zip(axes, models):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

fig.suptitle(
    f"Confusion Matrix Comparison Across Models (All Features, {year})",
    fontsize=16
)

fig.tight_layout()
fig.savefig(fig_path / "confusion_matrices_all_features.png", dpi=200, bbox_inches="tight")
plt.close(fig)


In [31]:
# Prints the classification report for each model

from sklearn.metrics import classification_report

classification_reports_all = {}

models_preds = {
    "Random Forest": rf_y_pred,
    "KNN": knn_y_pred,
    "Logistic Regression": lr_y_pred,
    "XGBoost": xgb_y_pred
}

for model, y_pred in models_preds.items():
    report = classification_report(y_test, y_pred, output_dict=True)
    classification_reports_all[model] = report

## Feature Engineering and Model Interpretability

In [32]:
# Finding the most important features for predicting diabetes

feature_importance = pd.Series(xgb_model.feature_importances_, index=X_train.columns)

feature_importance_sorted = feature_importance.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
feature_importance_sorted.plot(kind="bar", ax=ax)

ax.set_title("Feature Importance for Diabetes Prediction ")
ax.set_ylabel("Importance Score")

fig.tight_layout()
fig.savefig(fig_path / "feature_importance_xgb.png", dpi=200, bbox_inches="tight")
plt.close(fig)

top_features = feature_importance.sort_values(ascending=False).head(10).index

metrics["top_features"] = list(top_features)

In [33]:
# Keep only top 10 features
top_features_with_diabetes = top_features.to_list()
top_features_with_diabetes.append("Diabetes_Binary")

# Create cleaned diabetes df with top10 features
diabetes_cleaned_top10 = diabetes_cleaned_df[top_features_with_diabetes]


In [34]:
# Retrain models only using top 10 features
X_train_top10, X_train_scaled_top10, X_test_scaled_top10, y_train_top10, y_test_top10 = scale_and_split(diabetes_cleaned_top10)

# Using Random Forest - Train and Evaluate RF on top 10 Features
rf_model_top10, rf_y_pred_top10, rf_cm_top10  = train_and_evaluate_model("Random Forest", X_train_scaled_top10, X_test_scaled_top10, y_train_top10, y_test_top10)

# Using K-Nearest Neighbors - Train and Evaluate KNN on top 10 Features
knn_model_top10, knn_y_pred_top10, knn_cm_top10 = train_and_evaluate_model("KNN", X_train_scaled_top10, X_test_scaled_top10, y_train_top10, y_test_top10)

# Using Logistic Regression - Train and Evaluate KNN on top 10 Features
lr_model_top10, lr_y_pred_top10, lr_cm_top10 = train_and_evaluate_model("Logistic Regression",X_train_scaled_top10, X_test_scaled_top10, y_train_top10, y_test_top10)

# Using XGBoost - Train and Evaluate KNN on all top 10 Features
xgb_model_top10, xgb_y_pred_top10, xgb_cm_top10 = train_and_evaluate_model("XGBoost", X_train_scaled_top10, X_test_scaled_top10, y_train_top10, y_test_top10)

# Plot each model's confusion matrix side by side
fig, axes = plt.subplots(1, 4, figsize=(20,5))

models_top_10 = [
    ("Random Forest", rf_cm_top10),
    ("KNN", knn_cm_top10),
    ("Logistic Regression", lr_cm_top10),
    ("XGBoost", xgb_cm_top10)
]

for ax, (name, cm) in zip(axes, models_top_10):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

fig.suptitle(
    f"Confusion Matrix Comparison Across Models (Using Top 10 Features, {year})",
    fontsize=16
)

fig.savefig(fig_path / "confusion_matrices_top10_features.png", dpi=200, bbox_inches="tight")
plt.close(fig)


In [35]:
# Prints the classification report for each model

classification_reports_top10 = {}

models_top10_preds = {
    "Random Forest": rf_y_pred_top10,
    "KNN": knn_y_pred_top10,
    "Logistic Regression": lr_y_pred_top10,
    "XGBoost": xgb_y_pred_top10
}

for model, y_pred_top10 in models_top10_preds.items():
    report = classification_report(y_test, y_pred_top10, output_dict=True)
    classification_reports_top10[model] = report

In [36]:
# dump metrics for all features and top10

metrics["confusion_matrices"] = {
    "all_features": {
        "Random Forest": rf_cm,
        "KNN": knn_cm,
        "Logistic Regression": lr_cm,
        "XGBoost": xgb_cm
    },
    "top10_features": {
        "Random Forest": rf_cm_top10,
        "KNN": knn_cm_top10,
        "Logistic Regression": lr_cm_top10,
        "XGBoost": xgb_cm_top10
    }
}

metrics["classification_reports"] = {
    "all_features": classification_reports_all,
    "top10_features": classification_reports_top10
}

metrics["feature_importance"] = feature_importance_sorted.to_dict()
metrics["top_features"] = list(top_features)

In [37]:
# Plot ROV Curves

from sklearn.metrics import roc_curve, roc_auc_score

# Probabilities from top-10 models
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]
y_prob_knn = knn_model.predict_proba(X_test_scaled)[:, 1]
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
y_prob_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Random Forest
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)

# KNN
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_prob_knn)
auc_knn = roc_auc_score(y_test, y_prob_knn)

# Logistic Regression
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)

# XGBoost
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
auc_xgb = roc_auc_score(y_test, y_prob_xgb)

In [38]:

# Save figure
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.2f})', color='#4C72B0')
ax.plot(fpr_knn, tpr_knn, label=f'KNN (AUC = {auc_knn:.2f})', color='#55A868')
ax.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {auc_lr:.2f})', color='#C44E52')
ax.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {auc_xgb:.2f})', color='#8172B2')


# Random classifier line
ax.plot([0,1], [0,1], 'k--', label='Random Classifier')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves Comparison ({year})')
ax.legend(loc='lower right')
ax.grid(True)

fig.tight_layout()
fig.savefig(fig_path / "roc_curves.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# Update metrics

metrics["roc"] = {
    "Random Forest": (fpr_rf, tpr_rf, auc_rf),
    "KNN": (fpr_knn, tpr_knn, auc_knn),
    "Logistic Regression": (fpr_lr, tpr_lr, auc_lr),
    "XGBoost": (fpr_xgb, tpr_xgb, auc_xgb)
}

## Threshold Optimization for Final XGBoost Model



In [39]:
# Check different threshholds to lower false negatives on best model

from sklearn.metrics import confusion_matrix, recall_score, precision_score, accuracy_score

xgb_final = XGBClassifier(

              n_estimators=200,
              learning_rate=0.05,
              max_depth=5,
              scale_pos_weight=((len(y_train) - y_train.sum()) / y_train.sum()),
              random_state=42,
              n_jobs=-1
  )

xgb_final.fit(X_train_scaled, y_train)


y_prob_xgb_final = xgb_final.predict_proba(X_test_scaled)[:, 1]

thresholds = np.arange(0.1, 0.91, 0.05)
results = []


for t in thresholds:
    preds = (y_prob_xgb_final >= t).astype(int)

    recall = recall_score(y_test, preds)
    precision = precision_score(y_test, preds)
    accuracy = accuracy_score(y_test, preds)


    results.append({
        "Threshold": t,
        "Recall": recall,
        "Precision": precision,
        "Accuracy": accuracy
    })

threshold_df = pd.DataFrame(results)

metrics["threshold_tuning"] = threshold_df.to_dict()


In [40]:
# save plot for threshold tuning

fig, ax = plt.subplots(figsize=(8,5))

ax.plot(threshold_df["Threshold"], threshold_df["Recall"], label="Recall")
ax.plot(threshold_df["Threshold"], threshold_df["Precision"], label="Precision")
ax.plot(threshold_df["Threshold"], threshold_df["Accuracy"], label="Accuracy")

ax.set_title(f"Threshold Tuning - XGBoost ({year})")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.legend()

fig.tight_layout()
fig.savefig(fig_path / "threshold_tuning_xgb.png", dpi=200, bbox_inches="tight")
plt.close(fig)

In [41]:
# Dump all metrics
with open(data_path / 'metrics.pkl', 'wb') as f:
    pickle.dump(metrics, f)